In [ ]:
from dataclasses import dataclass
from enum import StrEnum
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools import tool

from langchain_ollama import ChatOllama


# =========================================================
# 1. CONTEXT
# =========================================================

@dataclass
class AgentContext:
    user_id: str
    role: str
    branch_code: str


# =========================================================
# 2. PERMISSIONS
# =========================================================

class Permission(StrEnum):
    VIEW_BRANCH_SCORE = "view_branch_score"
    VIEW_CUSTOMER_DATA = "view_customer_data"


# =========================================================
# 3. TOOLS
# =========================================================

@tool
def get_branch_score(branch_code: str) -> str:
    """
    Get the performance score of a bank branch.
    """

    # در پروژه واقعی:
    # Database / API

    return f"Branch {branch_code} score = 87.5"


@tool
def get_customer_data(customer_id: str) -> str:
    """
    Get customer information.
    """

    # در پروژه واقعی:
    # Database / API

    return f"Customer {customer_id}: Ali Ahmadi"


# =========================================================
# 4. TOOL → PERMISSION MAPPING
# =========================================================

TOOL_PERMISSIONS: dict[str, Permission] = {
    get_branch_score.name: Permission.VIEW_BRANCH_SCORE,
    get_customer_data.name: Permission.VIEW_CUSTOMER_DATA,
}


# =========================================================
# 5. AUTHORIZATION SERVICE
# =========================================================

def has_permission(
    context: AgentContext,
    permission: Permission,
) -> bool:

    # -----------------------------------------------------
    # Branch Score
    # -----------------------------------------------------

    if permission == Permission.VIEW_BRANCH_SCORE:

        return context.role in {
            "BranchManager",
            "RegionalManager",
            "HeadOffice",
        }

    # -----------------------------------------------------
    # Customer Data
    # -----------------------------------------------------

    if permission == Permission.VIEW_CUSTOMER_DATA:

        return context.role in {
            "RegionalManager",
            "HeadOffice",
        }

    # -----------------------------------------------------
    # Unknown permission
    # -----------------------------------------------------

    return False


# =========================================================
# 6. RESOURCE AUTHORIZATION
# =========================================================

def can_access_branch(
    context: AgentContext,
    requested_branch: str,
) -> bool:

    # مدیر شعبه فقط شعبه خودش را می‌بیند.

    if context.role == "BranchManager":
        return requested_branch == context.branch_code

    # مدیر منطقه و ستاد فعلاً دسترسی گسترده‌تر دارند.

    if context.role in {
        "RegionalManager",
        "HeadOffice",
    }:
        return True

    return False


# =========================================================
# 7. AUTHORIZATION MIDDLEWARE
# =========================================================

@wrap_tool_call
def authorization_middleware(
    request,
    handler: Callable,
):
    """
    Central authorization point.

    Middleware مسئول اجرای Business Rule نیست.
    فقط درخواست را به Authorization Service می‌فرستد.
    """

    # -----------------------------------------------------
    # Tool
    # -----------------------------------------------------

    tool_name = request.tool_call["name"]

    # -----------------------------------------------------
    # Permission مربوط به Tool
    # -----------------------------------------------------

    permission = TOOL_PERMISSIONS.get(tool_name)

    if permission is None:

        return ToolMessage(
            content="Access denied: unknown tool permission.",
            tool_call_id=request.tool_call["id"],
        )

    # -----------------------------------------------------
    # User Context
    # -----------------------------------------------------

    context = request.runtime.context

    print()
    print("========================================")
    print("AUTHORIZATION")
    print("========================================")

    print("User:", context.user_id)
    print("Role:", context.role)
    print("Tool:", tool_name)
    print("Permission:", permission)

    # =====================================================
    # Permission Check
    # =====================================================

    if not has_permission(
        context=context,
        permission=permission,
    ):

        print("PERMISSION DENIED")

        return ToolMessage(
            content="Access denied.",
            tool_call_id=request.tool_call["id"],
        )

    # =====================================================
    # Resource-level Authorization
    # =====================================================

    if permission == Permission.VIEW_BRANCH_SCORE:

        requested_branch = request.tool_call["args"]["branch_code"]

        if not can_access_branch(
            context=context,
            requested_branch=requested_branch,
        ):

            print("RESOURCE ACCESS DENIED")

            return ToolMessage(
                content=(
                    "Access denied: "
                    "You are not allowed to access this branch."
                ),
                tool_call_id=request.tool_call["id"],
            )

    # =====================================================
    # ACCESS GRANTED
    # =====================================================

    print("ACCESS GRANTED")

    return handler(request)


# =========================================================
# 8. MODEL
# =========================================================

model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)


# =========================================================
# 9. AGENT
# =========================================================

agent = create_agent(
    model=model,

    tools=[
        get_branch_score,
        get_customer_data,
    ],

    middleware=[
        authorization_middleware,
    ],

    context_schema=AgentContext,
)


# =========================================================
# 10. USER CONTEXT
# =========================================================

context = AgentContext(
    user_id="U1001",
    role="BranchManager",
    branch_code="101",
)


# =========================================================
# 11. REQUEST
# =========================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "امتیاز شعبه 101 را به من بگو.",
            }
        ]
    },
    context=context,
)


# =========================================================
# 12. RESULT
# =========================================================

print()
print("========================================")
print("FINAL RESPONSE")
print("========================================")

print(
    result["messages"][-1].content
)